In [ ]:
import openai
from neo4j import GraphDatabase

# Initialize the Neo4j database connection
uri = "bolt://localhost:7687"
user = "neo4j"
password = "your_password"
driver = GraphDatabase.driver(uri, auth=(user, password))

# Set your OpenAI API key
openai.api_key = "your-openai-api-key"

# Query the Neo4j Knowledge Graph
def query_neo4j(query: str) -> str:
    session = driver.session()
    result = session.run(query)
    result_data = [record for record in result]
    session.close()
    return str(result_data)

# Define the LLM interface to interact with OpenAI (GPT-3 or GPT-4)
def query_openai(prompt: str) -> str:
    response = openai.Completion.create(
        engine="gpt-4",  # Can use "gpt-3.5-turbo" as well
        prompt=prompt,
        max_tokens=150,
        temperature=0.7
    )
    return response.choices[0].text.strip()

# The Agentic Flow: Plan, Retrieve, Synthesize, Answer
def agentic_query_flow(user_query: str):
    # Step 1: Agent plans the query decomposition
    plan_prompt = f"""
    You are an intelligent agent. Given the following user query, break it down into actionable steps:
    User Query: "{user_query}"
    What steps are needed to gather the necessary information from a knowledge graph?
    """
    plan = query_openai(plan_prompt)
    print("Planning Step:", plan)

    # Step 2: Query the Neo4j Knowledge Graph (e.g., retrieve movie details)
    if "directed" in user_query.lower():  # For example, if the query is about who directed a movie
        movie_title = "Inception"
        neo4j_query = f"""
        MATCH (m:Movie {{title: "{movie_title}"}})<-[:DIRECTED]-(d:Person)
        RETURN d.name AS director
        """
        graph_data = query_neo4j(neo4j_query)

        # Step 3: Synthesize the final response
        synthesize_prompt = f"""
        Given the following data from the knowledge graph: {graph_data}
        Answer the following question:
        Who directed the movie '{movie_title}'?
        """
        final_answer = query_openai(synthesize_prompt)
        return final_answer

# Example user query
user_query = "Who directed Inception?"

# Execute the agentic flow
answer = agentic_query_flow(user_query)
print(f"Final Answer: {answer}")
